<a href="https://colab.research.google.com/github/artnight/Supply-Chain-SME/blob/main/01_generate_mock_data_to_db_and_excel.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
 !pip install faker holidays requests -q

import pandas as pd
import numpy as np
import random
import string
import os
from faker import Faker
import holidays
from datetime import datetime, timedelta

# ตั้งค่าความเป็นไทยและ Seed สำหรับควบคุมการสุ่ม
fake = Faker('th_TH')
np.random.seed(42)
random.seed(42)

# โฟลเดอร์ปลายทางสำหรับเก็บโมเดลและไฟล์จำลอง
OUTPUT_DIR = "mock_data"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# --- Shared config ---
N_PRODUCTS = 80
N_CUSTOMERS = 500
N_STORES = 10
N_WAREHOUSES = 3
N_PROMOTIONS = 30
START_DATE = datetime(2023, 1, 1)
END_DATE = datetime(2024, 12, 31)
DATE_RANGE = pd.date_range(START_DATE, END_DATE, freq='D')

# การจัดกลุ่มประเภทธุรกิจ (Taxonomies)
PRODUCT_CATEGORIES = ['beverages', 'snacks', 'dairy', 'frozen_food', 'personal_care', 'household', 'bakery', 'condiments']
STORE_TYPES = ['urban_large', 'suburban_medium', 'rural_small', 'convenience']
CUSTOMER_SEGS = ['high_value', 'regular', 'occasional', 'new']
WAREHOUSE_TYPES = ['central', 'regional_north', 'regional_south']

# ฟังก์ชันรัน Generator รหัสประจำตัว (ID)
def rand_id(prefix, n, width=3):
    return [f"{prefix}{str(i).zfill(width)}" for i in range(1, n+1)]

# เจเนอเรตไอดีสำหรับนำไปผูกความสัมพันธ์เชิงลึก (Relational Mapping)
product_ids = rand_id("SKU", N_PRODUCTS)
customer_ids = rand_id("CUST", N_CUSTOMERS)
store_ids = rand_id("STORE", N_STORES)
warehouse_ids = rand_id("WH", N_WAREHOUSES)
promo_ids = rand_id("PROMO", N_PROMOTIONS)

print("📊 [Cell 1] Config ready!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 67.1 MB/s eta 0:00:00
📊 [Cell 1] Config ready!


In [7]:
# WHY: product_taxonomies enables grouped/hierarchical forecasting.
# Base_price is used to compute revenue and gross margin.

category_map = {pid: random.choice(PRODUCT_CATEGORIES) for pid in product_ids}
prices = np.random.uniform(50.0, 500.0, size=N_PRODUCTS).round(2)
costs = (prices * np.random.uniform(0.4, 0.6, size=N_PRODUCTS)).round(2)

product_master = pd.DataFrame({
    'product_id': product_ids,
    'price': prices,
    'cost': costs,
    'product_taxonomies': [category_map[pid] for pid in product_ids],
    'unit_of_measure': np.random.choice(['pack', 'bottle', 'kg'], size=N_PRODUCTS),
    'shelf_life_days': np.random.choice([7, 14, 30, 90, 180], size=N_PRODUCTS)
})

# ตารางความยืดหยุ่นของราคา (Price Elasticity)
price_elasticity = pd.DataFrame({
    'product_id': product_ids,
    'elasticity_score': np.random.uniform(-2.5, -0.5, size=N_PRODUCTS).round(2)
})

product_master.to_csv(os.path.join(OUTPUT_DIR, "product_master.csv"), index=False)
price_elasticity.to_csv(os.path.join(OUTPUT_DIR, "price_elasticity.csv"), index=False)
print(f"💾 Product Tables Saved! Shape: {product_master.shape}")

💾 Product Tables Saved! Shape: (80, 6)


In [8]:
# WHY: store_taxonomies lets us train separate sub-models per store type.
# Region is used to join weather data (different regions = different weather).

store_master = pd.DataFrame({
    'store_id': store_ids,
    'store_taxonomies': [random.choice(STORE_TYPES) for _ in store_ids],
    'region': np.random.choice(['Bangkok', 'North', 'South', 'Isan'], size=N_STORES),
    'opening_year': np.random.choice(range(2015, 2023), size=N_STORES),
    'sqft_size': np.random.choice([1500, 3000, 5000, 10000], size=N_STORES)
})

store_master.to_csv(os.path.join(OUTPUT_DIR, "store_master.csv"), index=False)
print(f"💾 Store Master Saved! Shape: {store_master.shape}")

💾 Store Master Saved! Shape: (10, 5)


In [9]:
# WHY: customer_taxonomies feeds into RFM segmentation with prior labels.
# Is_loyalty_member is used as a feature weight in the uplift model.

customer_master = pd.DataFrame({
    'customer_id': customer_ids,
    'customer_taxonomies': [random.choice(CUSTOMER_SEGS) for _ in customer_ids],
    'acquisition_channel': np.random.choice(['Organic', 'FB Ads', 'TikTok', 'Walk-in'], size=N_CUSTOMERS),
    'is_loyalty_member': np.random.choice([True, False], p=[0.4, 0.6], size=N_CUSTOMERS),
    'registration_date': [fake.date_between(start_date='-3y', end_date='today').strftime('%Y-%m-%d') for _ in range(N_CUSTOMERS)],
    'province': [fake.province() for _ in range(N_CUSTOMERS)]
})

customer_master.to_csv(os.path.join(OUTPUT_DIR, "customer_master.csv"), index=False)
print(f"💾 Customer Master Saved! Shape: {customer_master.shape}")

💾 Customer Master Saved! Shape: (500, 6)


In [10]:
# WHY: warehouse_type defines capacity constraints for ROI calculation.

warehouse_master = pd.DataFrame({
    'warehouse_id': warehouse_ids,
    'warehouse_name': [f"WH_{t}" for t in WAREHOUSE_TYPES],
    'location': ['Bangkok', 'Chiang Mai', 'Chonburi'],
    'capacity_pallets': [50000, 20000, 30000],
    'has_cold_storage': [True, False, True]
})

warehouse_master.to_csv(os.path.join(OUTPUT_DIR, "warehouse_master.csv"), index=False)
print(f"💾 Warehouse Master Saved! Shape: {warehouse_master.shape}")

💾 Warehouse Master Saved! Shape: (3, 5)


In [11]:
# WHY: discount + structural date = promo_action feature for ML forecasting.

promo_records = []
for i, p_id in enumerate(promo_ids):
    start_dt = random.choice(DATE_RANGE)
    end_dt = start_dt + timedelta(days=random.randint(5, 20))
    promo_records.append({
        'promotion_id': p_id,
        'promotion_name': f"Campaign_{fake.word().capitalize()}",
        'discount_pct': random.choice([0.05, 0.10, 0.15, 0.20, 0.25, 0.30]),
        'product_id': random.choice(product_ids),
        'start_date': start_dt.strftime('%Y-%m-%d'),
        'end_date': end_dt.strftime('%Y-%m-%d'),
        'channel_cost': random.choice([1000, 2500, 5000, 10000])
    })

promotion_master = pd.DataFrame(promo_records)
promotion_master.to_csv(os.path.join(OUTPUT_DIR, "promotion_master.csv"), index=False)
print(f"💾 Promotion Master Saved! Shape: {promotion_master.shape}")

💾 Promotion Master Saved! Shape: (30, 7)


In [12]:
# WHY: External time-series features to handle seasonality spikes and anomalies.

th_holidays = holidays.Thailand(years=[2023, 2024])

holiday_data = []
weather_data = []

for dt in DATE_RANGE:
    dt_str = dt.strftime('%Y-%m-%d')

    # วันหยุด
    is_hol = 1 if dt in th_holidays or dt.strftime('%w') in ['0', '6'] else 0
    h_name = th_holidays.get(dt) if th_holidays.get(dt) else ('Weekend' if is_hol else 'Regular Day')
    holiday_data.append({'date': dt_str, 'holiday_name': h_name, 'is_holiday': is_hol})

    # สภาพอากาศ
    t = round(np.random.uniform(25.0, 39.0), 1)
    r = round(np.random.exponential(4.0) if np.random.rand() > 0.7 else 0.0, 1)
    weather_data.append({
        'date': dt_str,
        'temperature_celsius': t,
        'rainfall_mm': r,
        'weather_condition': 'Rainy' if r > 3.0 else ('Extreme Hot' if t > 36.0 else 'Normal')
    })

holiday_calendar = pd.DataFrame(holiday_data)
weather_data_df = pd.DataFrame(weather_data)

holiday_calendar.to_csv(os.path.join(OUTPUT_DIR, "holiday_calendar.csv"), index=False)
weather_data_df.to_csv(os.path.join(OUTPUT_DIR, "weather_data.csv"), index=False)
print("💾 Time-Series External Factors Saved!")

💾 Time-Series External Factors Saved!


In [14]:
# WHY: Core transactional fact table for baseline demand patterns.
# Fixed: Probability distribution matches the combined size of promo_ids and 'NO_PROMO'.

n_sales = 10000
sales_dates = [random.choice(DATE_RANGE) + timedelta(hours=random.randint(8, 21), minutes=random.randint(0, 59)) for _ in range(n_sales)]

# คำนวณการกระจายน้ำหนักความน่าจะเป็น (Probability Distribution) ให้สัมพันธ์กับจำนวนโปรโมชัน
n_promos = len(promo_ids)
if n_promos > 0:
    # เฉลี่ยสัดส่วน 30% ให้โปรโมชันทุกตัวเท่าๆ กัน และ 70% ให้ NO_PROMO
    promo_probabilities = [0.3 / n_promos] * n_promos + [0.7]
else:
    promo_probabilities = [1.0]

sales_transaction = pd.DataFrame({
    'po_id': rand_id("PO", n_sales, width=6),
    'datetime': [d.strftime('%Y-%m-%d %H:%M:%S') for d in sales_dates],
    'product_id': np.random.choice(product_ids, size=n_sales),
    'qty': np.random.choice([1, 2, 3, 4, 5], p=[0.5, 0.25, 0.15, 0.07, 0.03], size=n_sales),
    'customer_id': np.random.choice(customer_ids, size=n_sales),

    # ดึงค่า 확률ที่คำนวณไว้มาใส่ในช่อง p ให้ขนาด (Size) ตรงกันเป๊ะ
    'promotion_id': np.random.choice(promo_ids + ['NO_PROMO'], p=promo_probabilities, size=n_sales),

    'store_id': np.random.choice(store_ids, size=n_sales)
})

# ดึงราคาขายปัจจุบันมาแปะใส่ตารางธุรกรรม
sales_transaction['price'] = sales_transaction['product_id'].map(product_master.set_index('product_id')['price'])

sales_transaction.to_csv(os.path.join(OUTPUT_DIR, "sales_transaction.csv"), index=False)
print(f"💾 Sales Transaction Log Generated! Shape: {sales_transaction.shape}")

💾 Sales Transaction Log Generated! Shape: (10000, 8)


In [15]:
# WHY: Tracks actual logistic throughput to audit lead time bottlenecks.

n_moves = 3500
move_dates = [random.choice(DATE_RANGE) + timedelta(hours=random.randint(6, 18)) for _ in range(n_moves)]

stock_movement = pd.DataFrame({
    'movement_id': rand_id("MV", n_moves, width=6),
    'datetime': [d.strftime('%Y-%m-%d %H:%M:%S') for d in move_dates],
    'product_id': np.random.choice(product_ids, size=n_moves),
    'warehouse_id': np.random.choice(warehouse_ids, size=n_moves),
    'store_id': np.random.choice(store_ids, size=n_moves),
    'movement_type': np.random.choice(['INBOUND', 'TRANSFER_STORE', 'RETURNS'], p=[0.3, 0.6, 0.1], size=n_moves),
    'qty': np.random.randint(10, 100, size=n_moves)
})

stock_movement.to_csv(os.path.join(OUTPUT_DIR, "stock_movement.csv"), index=False)
print(f"💾 Stock Movement Log Generated! Shape: {stock_movement.shape}")

💾 Stock Movement Log Generated! Shape: (3500, 7)


In [16]:
# WHY: Quantifies procurement lead times and inventory supply lines.

n_purchases = 600
purchasing_order = pd.DataFrame({
    'po_id': rand_id("SPO", n_purchases, width=5),
    'supplier_id': np.random.choice(['SUP_BANGKOK_DIST', 'SUP_GLOBAL_IMPORT', 'SUP_LOCAL_SME'], size=n_purchases),
    'order_date': [random.choice(DATE_RANGE).strftime('%Y-%m-%d') for _ in range(n_purchases)],
    'delivery_date': [(random.choice(DATE_RANGE) + timedelta(days=random.randint(2, 7))).strftime('%Y-%m-%d') for _ in range(n_purchases)],
    'product_id': np.random.choice(product_ids, size=n_purchases),
    'qty': np.random.choice([100, 200, 500, 1000], size=n_purchases),
    'status': np.random.choice(['DELIVERED', 'PENDING', 'CANCELLED'], p=[0.85, 0.12, 0.03], size=n_purchases)
})

purchasing_order['cost'] = purchasing_order['product_id'].map(product_master.set_index('product_id')['cost'])

purchasing_order.to_csv(os.path.join(OUTPUT_DIR, "purchasing_order.csv"), index=False)
print(f"💾 Purchasing Order Log Generated! Shape: {purchasing_order.shape}")

💾 Purchasing Order Log Generated! Shape: (600, 8)


In [17]:
# WHY: Directly impacts net margin optimization by tracking deadstock waste.

n_wastes = 250
waste_log = pd.DataFrame({
    'waste_id': rand_id("WST", n_wastes, width=5),
    'datetime': [random.choice(sales_dates).strftime('%Y-%m-%d %H:%M:%S') for _ in range(n_wastes)],
    'product_id': np.random.choice(product_ids, size=n_wastes),
    'warehouse_id': np.random.choice(warehouse_ids, size=n_wastes),
    'qty': np.random.randint(1, 10, size=n_wastes),
    'reason': np.random.choice(['Expired Shelf Life', 'Damaged Packaging', 'Water Damage / Cold Loss'], p=[0.6, 0.3, 0.1], size=n_wastes)
})

waste_log.to_csv(os.path.join(OUTPUT_DIR, "waste_log.csv"), index=False)
print(f"💾 Waste Log Generated! Shape: {waste_log.shape}")
print("\n🔥 All 12 CSV Files Successfully Generated in 'mock_data/' folder!")

💾 Waste Log Generated! Shape: (250, 6)

🔥 All 12 CSV Files Successfully Generated in 'mock_data/' folder!


In [20]:
# WHY: Compilation stage for seamless tech stack handoff (SQL & Business Analysts)

data_map = {
    'customer_master': customer_master, 'holiday_calendar': holiday_calendar, 'price_elasticity': price_elasticity,
    'product_master': product_master, 'promotion_master': promotion_master, 'purchasing_order': purchasing_order,
    'sales_transaction': sales_transaction, 'stock_movement': stock_movement, 'store_master': store_master,
    'warehouse_master': warehouse_master, 'waste_log': waste_log, 'weather_data': weather_data_df
}

print("⚙️ Packing central data pipeline components...")

# 1. ยัดเข้าฐานข้อมูลเชิงสัมพันธ์ SQLite (.db)
db_conn = sqlite3.connect("supply_chain_sme_v2.db")
for table_name, target_df in data_map.items():
    target_df.to_sql(table_name, db_conn, if_exists='replace', index=False)
db_conn.close()
print("🗄️ Unified Relational SQLite Database Created -> 'supply_chain_sme_v2.db'")

# 2. ทำรายงานรวมเล่ม Excel สำหรับสายบริหาร (.xlsx)
with pd.ExcelWriter("supply_chain_data_40_products.xlsx", engine='openpyxl') as excel_writer:
    for tab_name, target_df in data_map.items():
        target_df.to_excel(excel_writer, sheet_name=tab_name, index=False)
print("📊 Comprehensive Business Spreadsheet Created -> 'supply_chain_data_40_products.xlsx'")
print("\n🌟 System Completed Smoothly!")

⚙️ Packing central data pipeline components...
🗄️ Unified Relational SQLite Database Created -> 'supply_chain_sme_v2.db'
📊 Comprehensive Business Spreadsheet Created -> 'supply_chain_data_40_products.xlsx'

🌟 System Completed Smoothly!
